# Bundle Clustering

## 1. Importing / Installing Packages

In [30]:
from __future__ import annotations

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import colors  # ✅ needed import
import seaborn as sns

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

from sklearn.cluster import DBSCAN
from hdbscan import HDBSCAN

from scipy.spatial import ConvexHull

from bayes_opt import BayesianOptimization

from typing import Dict, Optional, Sequence, Tuple, Iterable, Callable, cast

import geopandas as gpd
from shapely.wkt import loads as wkt_loads

from src.utils import read_csv_with_mapper

from dataclasses import dataclass, field

import glob

## 3. Data Import

### 3.1 Reading Filtered Header and midpoints

In [2]:
# Reading WellHeader excel file to dataframe
df_raw_wellheader = pd.read_csv(
    # r"C:\Users\apoorva.saxena\Desktop\Projects_AP\01. Ring Energy\Well Header\Header_Filtered.csv"
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Well Header\Header_Filtered.csv"
        ,usecols=["uwi", "well_name", "lease_name", "operator", "rsv_cat", "bench"] ,dtype={"uwi":str})

df_raw_wellheader = df_raw_wellheader[df_raw_wellheader["rsv_cat"]!="03PUD"].reset_index(drop=True).copy()

In [3]:
df_raw_wellheader

,uwi,well_name,lease_name,operator,rsv_cat,bench
0,42003002890200,UNIVERSITY BLOCK 9 4AB,UNIVERSITY BLOCK 9,Exxon Mobil,02PDNP,MISSISSIPPIAN
1,42003029150100,UNIVERSITY BLOCK 9 6AO,UNIVERSITY BLOCK 9,Exxon Mobil,02PDNP,SUB-WOODFORD
2,42003029190100,UNIVERSITY BLOCK 9 7AO,UNIVERSITY BLOCK 9,Exxon Mobil,02PDNP,SUB-WOODFORD
3,42003029190200,UNIVERSITY BLOCK 9 7AO,UNIVERSITY BLOCK 9,Exxon Mobil,02PDNP,SUB-WOODFORD
4,42003029720300,UNIVERSITY BLOCK 9 3AT,UNIVERSITY BLOCK 9,Exxon Mobil,01PDP,SUB-WOODFORD
...,...,...,...,...,...,...
1875,42501375990000,RED RAIDER 663 A 6H,RED RAIDER 663 A,Ring Energy,01PDP,SAN ANDRES
1876,42501376000000,RED RAIDER 663 B 7H,RED RAIDER 663 B,Ring Energy,01PDP,SAN ANDRES
1877,42501376010000,MF 732-733 1H,MF 732-733,Amtex Energy,01PDP,SAN ANDRES
1878,42501376030000,RED RAIDER 663 C 8H,RED RAIDER 663 C,Ring Energy,01PDP,SAN ANDRES


### 3.2 Reading Mid-Points data frame

In [4]:
df_midpoints = pd.read_csv(
    # r"C:\Users\apoorva.saxena\Desktop\Projects_AP\01. Ring Energy\Directional Surveys\Lateral_Midpoints.csv",
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Directional Surveys\Lateral_Midpoints.csv",
                           dtype={"uwi":str})

In [5]:
df_midpoints

,uwi,heel_lat,heel_lon,toe_lat,toe_lon,mid_Lat,mid_Lon
0,30025410040100,33.128186,-103.063231,33.139705,-103.063334,33.133946,-103.063283
1,30025421210000,33.113373,-103.069194,33.125204,-103.069167,33.119289,-103.069181
2,30025426220000,33.113377,-103.072581,33.125239,-103.073067,33.119308,-103.072824
3,30025428730000,33.113636,-103.063432,33.125201,-103.063393,33.119418,-103.063413
4,30025435920100,33.097298,-103.074979,33.111496,-103.075177,33.104397,-103.075078
...,...,...,...,...,...,...,...
2010,42501376070000,33.095593,-102.908611,33.124152,-102.907813,33.109872,-102.908212
2011,42501376080000,33.095416,-102.910903,33.124376,-102.910116,33.109896,-102.910510
2012,42501376090000,33.074192,-102.904032,33.095278,-102.903988,33.084735,-102.904010
2013,42501376100000,33.095262,-102.913369,33.109847,-102.913430,33.102554,-102.913399


### 3.3 Reading directional survey

In [6]:
def surveys_to_linestring_df(
    survey_df: pd.DataFrame,
    header_df: Optional[pd.DataFrame] = None,
    header_cols: Optional[Sequence[str]] = None,
    *,
    uwi_col: str = "uwi",
    lat_col: str = "latitude",
    lon_col: str = "longitude",
    md_col: str = "md",
    include_z: bool = False,
    z_col: str = "tvd",
    linestring_col: str = "geom_wkt",
    drop_single_point_wells: bool = True,
) -> pd.DataFrame:
    """
    Convert directional survey points into one WKT LINESTRING per well,
    optionally joined to a header table.

    The resulting DataFrame is ready to export to CSV / Parquet and can be
    read directly by Spotfire or QGIS (using `linestring_col` as the
    geometry column, CRS = EPSG:4326).

    Parameters
    ----------
    survey_df :
        Directional survey DataFrame with one row per survey station.
        Must contain at least:
        - uwi_col (default 'uwi')
        - lat_col (default 'latitude')  in decimal degrees
        - lon_col (default 'longitude') in decimal degrees
        - md_col  (default 'md') for sorting along the well path
        If `include_z=True`, also needs `z_col` (default 'tvd').

    header_df :
        Optional header DataFrame with one row per well (or several rows
        that can be deduplicated by `uwi_col`). This is joined after
        the LINESTRING is built.

    header_cols :
        Which columns from `header_df` to keep. If None, all columns in
        `header_df` are kept.

    uwi_col, lat_col, lon_col, md_col, z_col :
        Column names in `survey_df` / `header_df`. Adjust if your naming
        is different.

    include_z :
        If True, create a 3D LINESTRING Z (lon lat z). Otherwise create a
        2D LINESTRING (lon lat).

    linestring_col :
        Name of the output column containing WKT text.

    drop_single_point_wells :
        If True, wells with fewer than 2 valid survey points are dropped.
        If False, they are kept; in that case a POINT WKT is returned
        instead of a LINESTRING.

    Returns
    -------
    out_df :
        DataFrame with one row per well, containing:
        - uwi_col
        - linestring_col : WKT string (LINESTRING or LINESTRING Z)
        - any requested header columns (if header_df is provided)
    """
    # Work on a copy to avoid modifying the original
    df = survey_df.copy()

    # Ensure ordered along the well path
    df = df.sort_values([uwi_col, md_col])

    # Drop rows missing coordinates
    coord_cols = [lat_col, lon_col]
    if include_z:
        coord_cols.append(z_col)

    df = df.dropna(subset=coord_cols)

    records = []

    for uwi, grp in df.groupby(uwi_col, sort=False):
        if grp.empty:
            continue

        if len(grp) < 2 and drop_single_point_wells:
            # Not enough points to make a line – skip
            continue

        if include_z:
            coords = grp[[lon_col, lat_col, z_col]].to_numpy()
            coord_str = ", ".join(f"{x} {y} {z}" for x, y, z in coords)
            wkt = f"LINESTRING Z ({coord_str})"
        else:
            coords = grp[[lon_col, lat_col]].to_numpy()
            coord_str = ", ".join(f"{x} {y}" for x, y in coords)
            wkt = f"LINESTRING ({coord_str})"

        records.append({uwi_col: uwi, linestring_col: wkt})

    out_df = pd.DataFrame.from_records(records)

    # Attach header columns if provided
    if header_df is not None and not header_df.empty:
        if header_cols is None:
            header_use = header_df.copy()
        else:
            header_use = header_df[[uwi_col, *header_cols]].copy()

        header_use = header_use.drop_duplicates(subset=[uwi_col])
        out_df = out_df.merge(header_use, on=uwi_col, how="left")

    return out_df

In [7]:
df_directional_survey = pd.read_csv(
    # r"C:\Users\apoorva.saxena\Desktop\Projects_AP\01. Ring Energy\Directional Surveys\Directional_Survey.csv",
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Directional Surveys\Directional_Survey.csv",
    dtype={"uwi":str})

# Filter directional survey data to include only wells present in the raw well header data
df_directional_survey = df_directional_survey[df_directional_survey["uwi"].isin(df_raw_wellheader["uwi"].unique())].reset_index(drop=True).copy()

# Convert survey points to linestrings
df_survey_linestring = surveys_to_linestring_df(survey_df=df_directional_survey, header_df=df_raw_wellheader)

In [8]:
df_survey_linestring

,uwi,geom_wkt,well_name,lease_name,operator,rsv_cat,bench
0,30025410040100,"LINESTRING (-103.062995337 33.126830884, -103....",BROKEN SPOKE 2 STATE #001H,BROKEN SPOKE 2 STATE,Burk Royalty,01PDP,SAN ANDRES
1,30025421210000,"LINESTRING (-103.069252768 33.112247194, -103....",DOG BAR 11 FEE #002H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
2,30025426220000,"LINESTRING (-103.072580533 33.112225034, -103....",DOG BAR 11 FEE #003H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
3,30025428730000,"LINESTRING (-103.063123047 33.112295036, -103....",DOG BAR 11 FEE #001H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
4,30025435920100,"LINESTRING (-103.075096391 33.095841751, -103....",PINKMAN FEE #004H,PINKMAN FEE,Steward Energy II,02PA,SAN ANDRES
...,...,...,...,...,...,...,...
1875,42501375990000,"LINESTRING (-103.002091078 33.08165402, -103.0...",RED RAIDER 663 A 6H,RED RAIDER 663 A,Ring Energy,01PDP,SAN ANDRES
1876,42501376000000,"LINESTRING (-103.006275157 33.082538911, -103....",RED RAIDER 663 B 7H,RED RAIDER 663 B,Ring Energy,01PDP,SAN ANDRES
1877,42501376010000,"LINESTRING (-102.90883919 33.04218437, -102.90...",MF 732-733 1H,MF 732-733,Amtex Energy,01PDP,SAN ANDRES
1878,42501376030000,"LINESTRING (-103.006373059 33.082534816, -103....",RED RAIDER 663 C 8H,RED RAIDER 663 C,Ring Energy,01PDP,SAN ANDRES


### 3.4 Reading wps and average spacing data

In [9]:
df_wps = read_csv_with_mapper(
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\data\wps_summary.csv",
    col_map={"well_i": "uwi"}, dtype_map={"uwi": str})

df_avg_spacing = read_csv_with_mapper(
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\data\avg_spacing_summary.csv",
    col_map={"well_i": "uwi"}, dtype_map={"uwi": str}
)

In [10]:
df_avg_spacing

,uwi,avg_hz_spacing_ft,avg_vt_spacing_ft,neighbors_considered,adjacency_coverage_i_pct,avg_hz_spacing_ft_unw,avg_vt_spacing_ft_unw
0,30025410040100,933.507723,23.612500,1,1.000000,933.507723,23.612500
1,30025421210000,1427.412568,36.237720,2,1.000000,1431.014642,36.352500
2,30025426220000,1727.793704,47.650768,2,1.000000,1749.438382,48.404750
3,30025428730000,1754.494781,46.660500,1,0.997461,1754.494781,46.660500
4,30025435920100,NaN,NaN,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1910,42501376070000,1410.527894,28.376694,9,1.000000,1530.506228,31.309444
1911,42501376080000,1416.356465,41.255806,9,1.000000,1513.769545,45.707778
1912,42501376090000,1493.379759,37.495611,4,1.000000,1485.189132,37.869625
1913,42501376100000,1554.493574,24.620179,3,1.000000,1533.995306,25.895000


In [11]:
df_avg_spacing.merge(df_raw_wellheader, on="uwi", how="left")

,uwi,avg_hz_spacing_ft,avg_vt_spacing_ft,neighbors_considered,adjacency_coverage_i_pct,avg_hz_spacing_ft_unw,avg_vt_spacing_ft_unw,well_name,lease_name,operator,rsv_cat,bench
0,30025410040100,933.507723,23.612500,1,1.000000,933.507723,23.612500,BROKEN SPOKE 2 STATE #001H,BROKEN SPOKE 2 STATE,Burk Royalty,01PDP,SAN ANDRES
1,30025421210000,1427.412568,36.237720,2,1.000000,1431.014642,36.352500,DOG BAR 11 FEE #002H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
2,30025426220000,1727.793704,47.650768,2,1.000000,1749.438382,48.404750,DOG BAR 11 FEE #003H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
3,30025428730000,1754.494781,46.660500,1,0.997461,1754.494781,46.660500,DOG BAR 11 FEE #001H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
4,30025435920100,NaN,NaN,0,NaN,NaN,NaN,PINKMAN FEE #004H,PINKMAN FEE,Steward Energy II,02PA,SAN ANDRES
...,...,...,...,...,...,...,...,...,...,...,...,...
1910,42501376070000,1410.527894,28.376694,9,1.000000,1530.506228,31.309444,NaN,NaN,NaN,NaN,NaN
1911,42501376080000,1416.356465,41.255806,9,1.000000,1513.769545,45.707778,NaN,NaN,NaN,NaN,NaN
1912,42501376090000,1493.379759,37.495611,4,1.000000,1485.189132,37.869625,NaN,NaN,NaN,NaN,NaN
1913,42501376100000,1554.493574,24.620179,3,1.000000,1533.995306,25.895000,NaN,NaN,NaN,NaN,NaN


In [12]:
df_wps.merge(df_raw_wellheader, on="uwi", how="left")

,uwi,wps_cardinal,wps_iframe,wps_corridor,anisotropy_ratio,anisotropy_delta,azimuth_deg,mid_x,mid_y,well_name,lease_name,operator,rsv_cat,bench
0,30025410040100,3,3,2,1.0,0.0,358.510488,2.233137e+06,1.203348e+07,BROKEN SPOKE 2 STATE #001H,BROKEN SPOKE 2 STATE,Burk Royalty,01PDP,SAN ANDRES
1,30025421210000,3,3,3,1.0,0.0,359.055498,2.231430e+06,1.202812e+07,DOG BAR 11 FEE #002H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
2,30025426220000,3,3,3,1.0,0.0,356.969534,2.230315e+06,1.202810e+07,DOG BAR 11 FEE #003H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
3,30025428730000,3,3,3,1.0,0.0,359.103991,2.233195e+06,1.202820e+07,DOG BAR 11 FEE #001H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES
4,30025435920100,1,1,1,1.0,0.0,358.276644,2.229724e+06,1.202267e+07,PINKMAN FEE #004H,PINKMAN FEE,Steward Energy II,02PA,SAN ANDRES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2010,42501376070000,12,12,12,1.0,0.0,0.204329,2.280778e+06,1.202564e+07,NaN,NaN,NaN,NaN,NaN
2011,42501376080000,10,10,10,1.0,0.0,0.168816,2.280075e+06,1.202563e+07,NaN,NaN,NaN,NaN,NaN
2012,42501376090000,5,5,5,1.0,0.0,358.957836,2.282248e+06,1.201652e+07,NaN,NaN,NaN,NaN,NaN
2013,42501376100000,5,5,5,1.0,0.0,358.657432,2.279243e+06,1.202294e+07,NaN,NaN,NaN,NaN,NaN


## 4. Data Preprocessing

### 4.1 Filtering midpoints to only include those present in the wellheader

In [13]:
df_midpoints_filter = df_midpoints[df_midpoints["uwi"].isin(df_raw_wellheader["uwi"].unique())].reset_index(drop=True).copy()

## 5. Feature Engineering

In [14]:
# df_midpoints_filter.plot(kind='scatter',x='mid_Lat',y='mid_Lon',figsize=(19,8))

## 6. DBSCAN

### 6.1 Defining Functions

In [15]:
def haversine_distance(lon1, lat1, lon2, lat2,**kwargs):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    All args must be of equal length.

    """
    # convert degrees to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    # earth radius in km (match the DBSCAN constant)
    earth_radius_km = 6371.0088
    distance_km = earth_radius_km * c

    feet_per_km = 3280.84
    distance_ft = distance_km * feet_per_km

    return distance_ft

def dbscan_cluster(latitudes, longitudes, epsilon, min_samples, **kwargs):
    '''
    Function to perform DBSCAN clustering for given parameters.

    epsilon is provided in **feet**, converted to radians for the
    sklearn haversine metric.
    '''
    # constants
    kms_per_radian = 6371.0088        # km per radian on Earth's surface
    feet_per_km = 3280.84

    # convert epsilon from feet -> km -> radians
    epsilon_km = epsilon / feet_per_km
    epsilon_rad = epsilon_km / kms_per_radian

    dbscan = DBSCAN(
        eps=epsilon_rad,
        min_samples=min_samples,
        algorithm='ball_tree',
        metric='haversine',
        **kwargs
    )

    dbscan.fit(
        np.radians([x for x in zip(latitudes, longitudes)])
    )

    return pd.Series(dbscan.labels_)

def vertex_centroid_distance(latitudes,longitudes,**kwargs):
    '''
    Function to calculate the average distance from the vertices of a convex hull
    (derived from latitude x longitude pairs) to the centroid of said convex hull.
    
    Centroid is taken to be the unweighted average of all co-ordinate pairs.
    
    '''
    
    # co-ordinates of centre
    # take a simple average
    centre_long = longitudes.mean()
    centre_lats = latitudes.mean()
    
    # collapse two points into line
    if len(latitudes) < 3:
        distances = haversine_distance(
            longitudes,
            latitudes,
            centre_long,
            centre_lats,
            **kwargs).mean()
    
    else:
        # convex hull
        convex_hull = ConvexHull([x for x in zip(latitudes,longitudes)],**kwargs)

        # now get co-ordinates of vertices
        vertex_longs = longitudes.iloc[convex_hull.vertices]
        vertex_lats = latitudes.iloc[convex_hull.vertices]

        # now get
        distances = haversine_distance(
            vertex_longs,
            vertex_lats,
            centre_long,
            centre_lats,
            **kwargs).mean()

    # return average distance
    return distances.mean() if not np.isnan(distances) else 0.0

def calculate_average_values_in_disctionary(dictionary: Dict[int, float]) -> Optional[float]:
    """
    Return the arithmetic mean of the values in `dictionary` (positive).

    Parameters
    ----------
    dictionary : dict[int, float]
        Mapping from cluster_id -> average distance in feet.

    Returns
    -------
    float or None
        Positive average of the values, or None if the dict is empty.
    """
    if not dictionary:
        return None
    return float(np.mean(list(dictionary.values())))

def black_box_function(epsilon: float, min_samples: float) -> float:
    """
    Objective for Bayesian Optimization.

    Returns
    -------
    target : float
        Negative average cluster radius in feet
        (maximizing target == minimizing avg radius).
    """
    # 1) Prepare data
    df = df_midpoints_filter[['mid_Lat', 'mid_Lon']].copy()
    df.drop_duplicates(inplace=True)

    # 2) DBSCAN with epsilon in **feet**
    df['cluster'] = dbscan_cluster(
        latitudes=df['mid_Lat'],
        longitudes=df['mid_Lon'],
        epsilon=epsilon,
        min_samples=int(min_samples)
    )

    # 3) Per-cluster distances (feet)
    vertex_dist: Dict[int, float] = {}
    for cluster_id in df['cluster'].unique():
        df_cluster = df[df['cluster'] == cluster_id][['mid_Lat', 'mid_Lon']].copy()

        vertex_dist[cluster_id] = vertex_centroid_distance(
            latitudes=df_cluster['mid_Lat'],
            longitudes=df_cluster['mid_Lon']
        )

    # 4) Average radius across clusters (feet)
    avg_radius_ft = calculate_average_values_in_disctionary(vertex_dist)

    if avg_radius_ft is None or not np.isfinite(avg_radius_ft):
        # Neutral fallback if something degenerate happens
        return 0.0

    target = -avg_radius_ft
    return target

def build_bo_results_df(optimizer) -> pd.DataFrame:
    """
    Flatten optimizer.res into a DataFrame with
    iter, epsilon, min_samples, target, avg_radius_ft.
    """
    res = pd.DataFrame(optimizer.res)          # columns: ['target', 'params']
    params = res['params'].apply(pd.Series)    # split params into columns

    df = pd.concat([params, res['target']], axis=1)
    df.insert(0, 'iter', range(1, len(df) + 1))
    df['avg_radius_ft'] = -df['target']        # convert back to positive

    # nice column order
    return df[['iter', 'target', 'avg_radius_ft', 'epsilon', 'min_samples']]

def explore_linestring_clusters(
    df: pd.DataFrame,
    *,
    wkt_col: str = "geom_wkt",
    cluster_col: str = "cluster",
    crs: str = "EPSG:4326",
    tooltip_cols: Optional[Iterable[str]] = None,
    popup_cols: Optional[Iterable[str]] = None,
    tiles: str = "OpenStreetMap",
    legend: bool = True,
    style_kwds: Optional[dict] = None,
    **explore_kwargs,
):
    """
    Interactive map of well LINESTRINGs colored by cluster using GeoPandas .explore().

    Parameters
    ----------
    df :
        DataFrame with one row per well, containing:
        - WKT geometry column (wkt_col, e.g. 'geom_wkt')
        - cluster label column (cluster_col, e.g. 'cluster')
        plus any other columns you may want in tooltips/popups.

    wkt_col :
        Name of the WKT column.

    cluster_col :
        Name of the cluster column.

    crs :
        CRS of the coordinates. Typically 'EPSG:4326' for lat/lon.

    tooltip_cols :
        Columns to show when hovering over a line (lightweight).
        If None, tries ['uwi', 'well_name', 'cluster'] if present.

    popup_cols :
        Columns to show in a click popup (more detailed).
        If None, uses tooltip_cols.

    tiles :
        Basemap tiles; e.g. 'OpenStreetMap', 'CartoDB positron', etc.

    legend :
        Whether to show a cluster legend.

    style_kwds :
        Dict of style options for the linework, e.g. {'weight': 2}.

    **explore_kwargs :
        Passed through to gdf.explore(), e.g. zoom_start=10.

    Returns
    -------
    m, gdf :
        - m: folium.Map returned by GeoPandas .explore()
        - gdf: the GeoDataFrame used for plotting
    """
    # Build GeoDataFrame from WKT
    gdf = gpd.GeoDataFrame(
        df.drop(columns=[wkt_col]),
        geometry=df[wkt_col].apply(wkt_loads),
        crs=crs,
    )

    # Decide which columns to show in hover / popup
    if tooltip_cols is None:
        # Keep only those that exist
        default_tooltips = ["uwi", "well_name", cluster_col]
        tooltip_cols = [c for c in default_tooltips if c in gdf.columns]

    if popup_cols is None:
        popup_cols = tooltip_cols

    # Default line styling
    if style_kwds is None:
        style_kwds = {"weight": 2}

    m = gdf.explore(
        column=cluster_col,
        categorical=True,
        legend=legend,
        tiles=tiles,
        tooltip=list(tooltip_cols) if tooltip_cols else None,
        popup=list(popup_cols) if popup_cols else None,
        style_kwds=style_kwds,
        **explore_kwargs,
    )

    return m, gdf

### 6.2 Running Bayesian Optimization

In [16]:
# # Bounded region of parameter space
# pbounds = {'epsilon': (800, 2000), 'min_samples': (1, 2)}

# optimizer = BayesianOptimization(
#     f=black_box_function,
#     pbounds=pbounds,
#     random_state=0,
#     allow_duplicate_points=True
# )

# optimizer.maximize()

# results_df = build_bo_results_df(optimizer)

# results_df.sort_values('epsilon', ascending=True)

### 6.3 Getting and Plotting Final DB Cluster

In [17]:
# # Merge survey linestrings with cluster assignments
# df_survey_linestring_cluster = df_survey_linestring.merge(df_with_clusters[["uwi", "cluster"]], on="uwi", how="left").reset_index(drop=True).copy()

# gdf = gpd.GeoDataFrame(
#     df_survey_linestring_cluster.drop(columns=['geom_wkt']),
#     geometry=df_survey_linestring_cluster['geom_wkt'].apply(wkt_loads),
#     crs="EPSG:4326"
# )

# # Option A: shapefile (folder with .shp/.dbf/.shx/.prj)
# gdf.to_file(r"C:\Users\apoorva.saxena\Desktop\Projects_AP\01. Ring Energy\shapefiles\well_lines.shp")

# # Option B: GeoPackage (single file)
# # gdf.to_file("well_lines.gpkg", layer="well_lines", driver="GPKG")

## 7. DBSCAN - Class

### 7.1 Defining DBSCAN class

In [18]:
ScoreFn = Callable[[np.ndarray, pd.DataFrame], float]


@dataclass
class LeaseAwareBucketedWellDBSCAN:
    """
    Lease-aware DBSCAN bundling with spacing-based buckets + Bayesian optimization.

    High-level idea
    ---------------
    - Each well has:
        * Coordinates (midpoint lat/lon) from `midpoints_df`
        * Per-well spacing metric (e.g. avg horizontal spacing in ft) from `spacing_df`
        * Lease name from `header_df` (e.g. DSU / unit / lease)

    - We first compute a lease-level spacing scale:
        * For each lease, take the median of per-well spacing (e.g. median(avg_hz_spacing_ft)).
        * Every well in that lease inherits this `lease_median_spacing_ft`.

    - Then we define *spacing buckets* based on this lease-level spacing metric:
        * Example: 0–500 ft, 500–1000 ft, 1000–2000 ft (via `spacing_bins_ft`)
        * Or equal-width buckets of size `bucket_width_ft`.

      → So leases with similar overall spacing fall into the same bucket,
        and all wells in that lease share the same bucket.

    - For each spacing bucket, we:
        * Run Bayesian optimization over DBSCAN hyperparameters:
            - eps_ft = eps_factor * median_spacing_in_bucket
            - min_samples ∈ [min_samples_bounds]
        * Run DBSCAN with haversine metric on that bucket’s wells.
        * Offset labels so cluster IDs are globally unique across buckets.

    - The result is a DataFrame with:
        * All original midpoints and header columns
        * Per-well spacing, lease_median_spacing_ft
        * `spacing_bucket_id`
        * `cluster_id_global` (DBSCAN cluster, -1 = noise)

    Inputs
    ------
    midpoints_df:
        One row per well, with at least:
          - uwi_col (e.g. 'well_i')
          - lat_col, lon_col (latitude/longitude in degrees)
    spacing_df:
        Per-well spacing summary, with at least:
          - spacing_well_col (same logical ID as uwi_col)
          - spacing_col (e.g. 'avg_hz_spacing_ft' in feet)
    header_df:
        Header table with lease info, with at least:
          - header_uwi_col (same logical ID as uwi_col)
          - lease_col (e.g. 'LeaseName' or 'Lease')

    Parameters
    ----------
    uwi_col:
        Well identifier in midpoints_df (e.g. 'well_i').
    spacing_well_col:
        Well identifier in spacing_df (e.g. 'well_i').
    header_uwi_col:
        Well identifier in header_df (e.g. 'well_i').
    lease_col:
        Lease name column in header_df (e.g. 'LeaseName').
    spacing_col:
        Per-well spacing metric column in spacing_df
        (e.g. 'avg_hz_spacing_ft').
    lease_spacing_col:
        Name for the derived lease-level spacing column
        (default 'lease_median_spacing_ft').

    bucket_width_ft:
        If provided and spacing_bins_ft is None:
          - Equal-width buckets of this size based on lease-level spacing:
              bucket_id = floor(lease_spacing / bucket_width_ft).
        If None and spacing_bins_ft is also None:
          - All wells share a single bucket (global clustering).

    spacing_bins_ft:
        Optional explicit bin edges in feet for lease-level spacing.
        Example: [0, 500, 1000, 2000, np.inf] will create buckets
        [0,500), [500,1000), [1000,2000), [2000,inf).

        If provided, takes precedence over bucket_width_ft.

    min_bucket_size:
        Minimum number of wells in a spacing bucket required to run
        Bayesian optimization + DBSCAN. Buckets with fewer wells are skipped
        and those wells remain labeled as noise (-1).

    eps_factor_bounds:
        Bounds for eps_factor in Bayesian optimization:
            eps_ft = eps_factor * median_spacing_in_bucket
        e.g. (0.6, 1.4) says "search eps between 0.6× and 1.4× median spacing".

    min_samples_bounds:
        Integer bounds for min_samples in Bayesian optimization.

    random_state:
        Seed for BayesianOptimization.

    objective_fn:
        Optional custom scoring:
            objective_fn(labels: np.ndarray, bucket_df: pd.DataFrame) -> float
        If None, the default score is number of wells assigned to clusters
        (labels != -1).

    Attributes after fit_predict
    ----------------------------
    bucket_params_:
        Dict[bucket_id -> dict of best hyperparams]:
          {
              bucket_id: {
                  "eps_ft": ...,
                  "min_samples": ...,
                  "score": ...,
                  "n_wells": ...,
                  "n_clusters": ...,
                  "n_clustered": ...
              },
              ...
          }
    """

    midpoints_df: pd.DataFrame
    spacing_df: pd.DataFrame
    header_df: pd.DataFrame

    uwi_col: str = "well_i"
    spacing_well_col: str = "well_i"
    header_uwi_col: str = "well_i"
    lease_col: str = "lease_name"

    lat_col: str = "latitude"
    lon_col: str = "longitude"
    spacing_col: str = "avg_hz_spacing_ft"
    lease_spacing_col: str = "lease_median_spacing_ft"

    bucket_width_ft: Optional[float] = 500.0
    spacing_bins_ft: Optional[Sequence[float]] = None
    min_bucket_size: int = 10

    eps_factor_bounds: Tuple[float, float] = (0.6, 1.4)
    min_samples_bounds: Tuple[int, int] = (1, 2)

    random_state: int = 0
    objective_fn: Optional[ScoreFn] = None

    # NEW: hard-ish cap on bundle radius (in feet).
    # If None -> no post-splitting; you get raw DBSCAN clusters.
    max_bundle_radius_ft: Optional[float] = None

    # If True, assign unique cluster IDs to noise wells (-1),
    # turning them into singleton clusters with their own IDs.
    assign_unique_ids_to_noise: bool = False

    bucket_params_: Dict[int, Dict[str, float]] = field(
        init=False, default_factory=dict
    )

    # ------------------------------------------------------------------ #
    # Internal helpers
    # ------------------------------------------------------------------ #
    def _prepare_dataframe(self) -> pd.DataFrame:
        """
        Join midpoints + spacing + header, and compute lease-level spacing.

        Returns a DataFrame with:
          - all midpoints columns
          - header lease_col
          - per-well spacing_col
          - lease_spacing_col (median per lease)
        """
        # spacing_df + header_df -> per-well spacing + lease
        spacing_with_lease = self.spacing_df.merge(
            self.header_df[[self.header_uwi_col, self.lease_col]],
            left_on=self.spacing_well_col,
            right_on=self.header_uwi_col,
            how="left",
        )

        # Lease-level spacing metric (median)
        lease_stats = (
            spacing_with_lease
            .dropna(subset=[self.lease_col, self.spacing_col])
            .groupby(self.lease_col, as_index=False)
            .agg(**{self.lease_spacing_col: (self.spacing_col, "median")})
        )

        # midpoints + header -> get lease for each midpoint well
        df = self.midpoints_df.merge(
            self.header_df[[self.header_uwi_col, self.lease_col]],
            left_on=self.uwi_col,
            right_on=self.header_uwi_col,
            how="left",
        )

        # Add per-well spacing (can be NaN for some wells)
        df = df.merge(
            self.spacing_df[[self.spacing_well_col, self.spacing_col]],
            left_on=self.uwi_col,
            right_on=self.spacing_well_col,
            how="left",
        )

        # Add lease-level spacing metric
        df = df.merge(
            lease_stats[[self.lease_col, self.lease_spacing_col]],
            on=self.lease_col,
            how="left",
        )

        return df

    def _assign_buckets(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Assign spacing buckets based on lease-level spacing (lease_spacing_col).

        - If spacing_bins_ft is provided:
            use pd.cut with those edges on lease_spacing_col.
        - Else if bucket_width_ft is None:
            put everyone into bucket 0 (single global bucket).
        - Else:
            bucket_id = floor(lease_spacing_col / bucket_width_ft).
        """
        df = df.copy()

        if self.spacing_bins_ft is not None:
            bucket_codes = pd.cut(
                df[self.lease_spacing_col],
                bins=self.spacing_bins_ft,
                right=False,   # [low, high)
                labels=False,  # 0,1,2,...
            )
            df["spacing_bucket_id"] = bucket_codes.astype("Int64")
            return df

        if self.bucket_width_ft is None:
            df["spacing_bucket_id"] = 0
            return df

        bucket = (
            df[self.lease_spacing_col] / self.bucket_width_ft
        ).floordiv(1).astype("Int64")
        df["spacing_bucket_id"] = bucket
        return df

    def _greedy_radius_bundles_haversine(
        self,
        coords_deg: np.ndarray,
        max_radius_ft: float,
    ) -> np.ndarray:
        """
        Greedy 'ball' splitting inside one cluster.

        Parameters
        ----------
        coords_deg : array (n, 2)
            Latitude, longitude in degrees for wells in ONE cluster.
        max_radius_ft : float
            Max radius (feet) from the chosen seed to include in a bundle.

        Returns
        -------
        bundle_ids : np.ndarray of shape (n,)
            Integer bundle ID per point: 0,1,2,... (local to this cluster).
        """
        n = coords_deg.shape[0]
        if n == 0:
            return np.array([], dtype=int)
        if n == 1:
            return np.array([0], dtype=int)

        # Convert to radians once
        coords_rad = np.radians(coords_deg)
        R_earth_ft = 6371008.8 * 3.28084  # mean Earth radius in feet

        remaining = np.arange(n)
        bundle_ids = np.full(n, -1, dtype=int)
        next_bundle_id = 0

        while remaining.size > 0:
            seed_idx = remaining[0]
            seed = coords_rad[seed_idx]
            lat1 = seed[0]
            lon1 = seed[1]

            lat2 = coords_rad[remaining, 0]
            lon2 = coords_rad[remaining, 1]

            dlat = lat2 - lat1
            dlon = lon2 - lon1

            a = (
                np.sin(dlat / 2.0) ** 2
                + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
            )
            # numerical safety
            a = np.clip(a, 0.0, 1.0)
            c = 2.0 * np.arcsin(np.sqrt(a))
            dist_ft = R_earth_ft * c

            in_ball_mask = dist_ft <= max_radius_ft
            in_ball_idx = remaining[in_ball_mask]

            bundle_ids[in_ball_idx] = next_bundle_id
            next_bundle_id += 1

            # remove assigned wells from remaining set
            remaining = remaining[~in_ball_mask]

        return bundle_ids

    def _split_clusters_by_radius(
        self,
        df: pd.DataFrame,
        cluster_col: str,
    ) -> np.ndarray:
        """
        Post-process clusters: for each DBSCAN cluster, split it into
        smaller bundles such that each bundle fits inside a ball of
        radius `max_bundle_radius_ft` around some seed well.

        Returns
        -------
        new_labels : np.ndarray
            New global cluster labels after radius-based splitting.
        """
        if self.max_bundle_radius_ft is None:
            # nothing to do
            return df[cluster_col].to_numpy()

        max_r = float(self.max_bundle_radius_ft)
        labels = df[cluster_col].to_numpy().copy()

        # find next available cluster ID for newly created bundles
        existing = labels[labels >= 0]
        if existing.size == 0:
            return labels

        next_label = int(existing.max()) + 1

        for cid in sorted(np.unique(existing)):
            mask = labels == cid
            idx_cluster = np.where(mask)[0]

            if idx_cluster.size <= 1:
                continue

            coords_deg = df.loc[idx_cluster, [self.lat_col, self.lon_col]].to_numpy()

            # local bundle ids 0,1,2,... within this cluster
            bundle_ids = self._greedy_radius_bundles_haversine(coords_deg, max_r)
            unique_bundles = np.unique(bundle_ids)

            # if everything fits into one ball, keep original cluster id
            if unique_bundles.size == 1:
                continue

            # otherwise, keep cid for the first bundle and assign new IDs for others
            # (order is arbitrary but deterministic via sorted)
            for j, b in enumerate(sorted(unique_bundles)):
                idx_bundle = idx_cluster[bundle_ids == b]
                if j == 0:
                    # keep original cid for this first bundle
                    continue
                labels[idx_bundle] = next_label
                next_label += 1

        return labels

    @staticmethod
    def _dbscan_labels(
        lat: np.ndarray,
        lon: np.ndarray,
        epsilon_ft: float,
        min_samples: int,
    ) -> np.ndarray:
        """
        Run DBSCAN with haversine metric on lat/lon points.
        Epsilon is given in feet.
        """
        # Convert epsilon in feet to radians for haversine metric
        kms_per_radian = 6371.0088
        feet_per_km = 3280.84
        epsilon_km = epsilon_ft / feet_per_km
        eps_rad = epsilon_km / kms_per_radian

        coords_rad = np.vstack([np.radians(lat), np.radians(lon)]).T

        model = DBSCAN(
            eps=eps_rad,
            min_samples=min_samples,
            metric="haversine",
        )
        labels = model.fit_predict(coords_rad)
        return labels

    def _run_dbscan_for_bucket(
        self, bucket_df: pd.DataFrame
    ) -> Tuple[np.ndarray, float, int, float]:
        """
        Bayesian optimization over eps & min_samples for a single bucket.

        Returns
        -------
        labels_best : np.ndarray
        best_eps_ft : float
        best_min_samples : int
        best_score : float
        """
        from math import isfinite

        lat = bucket_df[self.lat_col].values
        lon = bucket_df[self.lon_col].values

        # Use per-well spacing for scaling; fall back to lease-level if needed
        spacing_vals = bucket_df[self.spacing_col].dropna().values
        if len(spacing_vals) == 0:
            spacing_vals = bucket_df[self.lease_spacing_col].dropna().values

        if len(spacing_vals) == 0:
            # No spacing info at all -> skip clustering
            n = len(bucket_df)
            return np.full(n, -1, dtype=int), float("nan"), 1, 0.0

        median_spacing = float(np.median(spacing_vals))
        if not isfinite(median_spacing) or median_spacing <= 0:
            n = len(bucket_df)
            return np.full(n, -1, dtype=int), float("nan"), 1, 0.0

        eps_low = self.eps_factor_bounds[0] * median_spacing
        eps_high = self.eps_factor_bounds[1] * median_spacing

        def objective(eps_factor: float, min_samples: float) -> float:
            eps_ft = eps_factor * median_spacing
            min_samp_int = max(1, int(round(min_samples)))

            labels = self._dbscan_labels(
                lat=lat,
                lon=lon,
                epsilon_ft=eps_ft,
                min_samples=min_samp_int,
            )

            if self.objective_fn is not None:
                return float(self.objective_fn(labels, bucket_df))

            # Default: maximize number of wells in non-noise clusters
            return float((labels != -1).sum())

        pbounds = {
            "eps_factor": (self.eps_factor_bounds[0], self.eps_factor_bounds[1]),
            "min_samples": (self.min_samples_bounds[0], self.min_samples_bounds[1]),
        }

        optimizer = BayesianOptimization(
            f=objective,
            pbounds=pbounds,
            random_state=self.random_state,
            allow_duplicate_points=True,
        )
        optimizer.maximize()

        best_params = optimizer.max["params"]
        best_score = float(optimizer.max["target"])

        best_eps_ft = best_params["eps_factor"] * median_spacing
        best_min_samples = max(1, int(round(best_params["min_samples"])))

        labels_best = self._dbscan_labels(
            lat=lat,
            lon=lon,
            epsilon_ft=best_eps_ft,
            min_samples=best_min_samples,
        )

        return labels_best, best_eps_ft, best_min_samples, best_score

    # ------------------------------------------------------------------ #
    # Public API
    # ------------------------------------------------------------------ #
    def preview_buckets(self) -> pd.DataFrame:
        """
        Return a small summary table of spacing_bucket_id distributions
        BEFORE running clustering.

        Useful to choose bucket_width_ft / spacing_bins_ft / min_bucket_size.
        """
        df = self._prepare_dataframe()
        df = self._assign_buckets(df)

        summary = (
            df.groupby("spacing_bucket_id")
            .agg(
                n_wells=(self.lease_spacing_col, "size"),
                lease_spacing_min=(self.lease_spacing_col, "min"),
                lease_spacing_max=(self.lease_spacing_col, "max"),
                lease_spacing_mean=(self.lease_spacing_col, "mean"),
            )
            .sort_index()
        )
        return summary

    def fit_predict(self) -> pd.DataFrame:
        """
        Run lease-aware, spacing-bucketed DBSCAN with Bayesian optimization.

        Returns
        -------
        df_out : DataFrame
            Original midpoints joined with header + spacing, plus:
              - lease_spacing_col (lease-level median spacing)
              - spacing_bucket_id
              - cluster_id_global (DBSCAN cluster, -1 = noise)
        """
        df = self._prepare_dataframe()
        df = self._assign_buckets(df)

        global_labels = np.full(len(df), -1, dtype=int)
        global_cluster_offset = 0
        bucket_params: Dict[int, Dict[str, float]] = {}

        for bucket_id, bucket_df in df.groupby("spacing_bucket_id"):
            if pd.isna(bucket_id):
                continue

            n_bucket = len(bucket_df)
            if n_bucket < self.min_bucket_size:
                continue

            (
                labels_local,
                best_eps_ft,
                best_min_samples,
                best_score,
            ) = self._run_dbscan_for_bucket(bucket_df)

            mask_clustered = labels_local != -1
            unique_local = np.unique(labels_local[mask_clustered])
            n_clusters = len(unique_local)
            n_clustered = int(mask_clustered.sum())

            if n_clusters == 0:
                continue

            # Map bucket-local labels to global IDs
            label_map = {
                local: (global_cluster_offset + i)
                for i, local in enumerate(sorted(unique_local))
            }
            remapped = np.where(
                mask_clustered,
                np.vectorize(label_map.get)(labels_local),
                -1,
            )

            global_labels[bucket_df.index.values] = remapped
            global_cluster_offset += n_clusters

            bucket_params[int(bucket_id)] = {
                "eps_ft": float(best_eps_ft),
                "min_samples": float(best_min_samples),
                "score": float(best_score),
                "n_wells": float(n_bucket),
                "n_clusters": float(n_clusters),
                "n_clustered": float(n_clustered),
            }

        df_out = df.copy()

        # Raw DBSCAN labels (before radius-based splitting)
        df_out["cluster_id_dbscan"] = global_labels

        # Apply radius-based splitting if requested
        if self.max_bundle_radius_ft is not None:
            labels = self._split_clusters_by_radius(
                df_out,
                cluster_col="cluster_id_dbscan",
            )
        else:
            labels = global_labels.copy()

        # Optionally turn noise (-1) into unique singleton cluster IDs
        if self.assign_unique_ids_to_noise:
            labels = labels.copy()
            noise_mask = labels == -1

            if noise_mask.any():
                # Start after the max existing non-noise label
                non_noise = labels[labels >= 0]
                start_id = int(non_noise.max()) + 1 if non_noise.size > 0 else 0

                noise_idx = np.where(noise_mask)[0]
                labels[noise_idx] = np.arange(
                    start_id, start_id + noise_idx.size, dtype=int
                )

        df_out["cluster_id_global"] = labels
        self.bucket_params_ = bucket_params

        return df_out

In [ ]:
db_lease = LeaseAwareBucketedWellDBSCAN(
    midpoints_df=df_midpoints_filter,
    spacing_df=df_avg_spacing,
    header_df=df_raw_wellheader,
    uwi_col="uwi",
    spacing_well_col="uwi",
    header_uwi_col="uwi",
    lease_col="lease_name",
    lat_col="mid_Lat",
    lon_col="mid_Lon",
    spacing_col="avg_hz_spacing_ft",
    # bucket_width_ft=500.0,
    spacing_bins_ft=list(np.arange(0, 5000, 100)),
    min_bucket_size=1,
    eps_factor_bounds=(0.6, 1.4),
    min_samples_bounds=(1, 2),
    random_state=42,
    max_bundle_radius_ft=1320.0,
    assign_unique_ids_to_noise=True
)

In [65]:
db_lease.preview_buckets()

,n_wells,lease_spacing_min,lease_spacing_max,lease_spacing_mean
spacing_bucket_id,,,,
0,21,10.079829,92.992722,32.684537
1,2,145.512404,145.512404,145.512404
2,3,226.341605,236.067348,232.046146
3,12,302.925638,391.982874,357.057652
4,13,415.918590,488.016639,463.512770
5,6,535.629615,551.103002,545.157793
6,17,624.014426,695.527183,668.390180
7,7,702.501001,783.715760,741.305246
8,31,824.782608,893.636291,865.809488


In [66]:
df_db_clusters_lease = db_lease.fit_predict()

|   iter    |  target   | eps_fa... | min_sa... |
-------------------------------------------------
| 1         | 0.0       | 0.8996    | 1.951     |
| 2         | 2.0       | 1.186     | 1.599     |
| 3         | 21.0      | 0.7248    | 1.156     |
| 4         | 0.0       | 0.6465    | 1.866     |
| 5         | 2.0       | 1.081     | 1.708     |
| 6         | 21.0      | 0.6       | 1.0       |
| 7         | 21.0      | 1.006     | 1.0       |
| 8         | 21.0      | 1.4       | 1.0       |
| 9         | 21.0      | 0.7896    | 1.0       |
| 10        | 21.0      | 1.227     | 1.0       |
| 11        | 21.0      | 0.6       | 1.222     |
| 12        | 21.0      | 0.6       | 1.113     |
| 13        | 21.0      | 0.916     | 1.12      |
| 14        | 21.0      | 1.108     | 1.086     |
| 15        | 21.0      | 1.4       | 1.114     |
| 16        | 21.0      | 1.083     | 1.152     |
| 17        | 21.0      | 0.8167    | 1.301     |
| 18        | 21.0      | 1.273     | 1.107     |


In [67]:
df_db_clusters_lease

,uwi,heel_lat,heel_lon,toe_lat,toe_lon,mid_Lat,mid_Lon,lease_name,avg_hz_spacing_ft,lease_median_spacing_ft,spacing_bucket_id,cluster_id_dbscan,cluster_id_global
0,30025410040100,33.128186,-103.063231,33.139705,-103.063334,33.133946,-103.063283,BROKEN SPOKE 2 STATE,933.507723,1183.017361,11,140,140
1,30025421210000,33.113373,-103.069194,33.125204,-103.069167,33.119289,-103.069181,DOG BAR 11 FEE,1427.412568,1727.793704,17,920,920
2,30025426220000,33.113377,-103.072581,33.125239,-103.073067,33.119308,-103.072824,DOG BAR 11 FEE,1727.793704,1727.793704,17,920,920
3,30025428730000,33.113636,-103.063432,33.125201,-103.063393,33.119418,-103.063413,DOG BAR 11 FEE,1754.494781,1727.793704,17,921,921
4,30025435920100,33.097298,-103.074979,33.111496,-103.075177,33.104397,-103.075078,PINKMAN FEE,NaN,NaN,<NA>,-1,1256
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1875,42501375990000,33.080080,-103.003140,33.066330,-103.003103,33.073205,-103.003122,RED RAIDER 663 A,1553.023535,1553.023535,15,804,804
1876,42501376000000,33.080256,-103.005285,33.066333,-103.005288,33.073295,-103.005287,RED RAIDER 663 B,1429.579045,1429.579045,14,649,649
1877,42501376010000,33.043411,-102.905997,33.047070,-102.897379,33.045240,-102.901688,MF 732-733,NaN,NaN,<NA>,-1,1443
1878,42501376030000,33.080130,-103.007432,33.062673,-103.007415,33.071402,-103.007424,RED RAIDER 663 C,1452.670508,1452.670508,14,649,649


In [68]:
df_survey_linestring_with_cluster = df_survey_linestring.merge(df_db_clusters_lease[[ "uwi", "avg_hz_spacing_ft", "cluster_id_global"]], on="uwi", how="left").reset_index(drop=True).copy()

In [69]:
df_survey_linestring_with_cluster

,uwi,geom_wkt,well_name,lease_name,operator,rsv_cat,bench,avg_hz_spacing_ft,cluster_id_global
0,30025410040100,"LINESTRING (-103.062995337 33.126830884, -103....",BROKEN SPOKE 2 STATE #001H,BROKEN SPOKE 2 STATE,Burk Royalty,01PDP,SAN ANDRES,933.507723,140
1,30025421210000,"LINESTRING (-103.069252768 33.112247194, -103....",DOG BAR 11 FEE #002H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES,1427.412568,920
2,30025426220000,"LINESTRING (-103.072580533 33.112225034, -103....",DOG BAR 11 FEE #003H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES,1727.793704,920
3,30025428730000,"LINESTRING (-103.063123047 33.112295036, -103....",DOG BAR 11 FEE #001H,DOG BAR 11 FEE,Burk Royalty,01PDP,SAN ANDRES,1754.494781,921
4,30025435920100,"LINESTRING (-103.075096391 33.095841751, -103....",PINKMAN FEE #004H,PINKMAN FEE,Steward Energy II,02PA,SAN ANDRES,NaN,1256
...,...,...,...,...,...,...,...,...,...
1875,42501375990000,"LINESTRING (-103.002091078 33.08165402, -103.0...",RED RAIDER 663 A 6H,RED RAIDER 663 A,Ring Energy,01PDP,SAN ANDRES,1553.023535,804
1876,42501376000000,"LINESTRING (-103.006275157 33.082538911, -103....",RED RAIDER 663 B 7H,RED RAIDER 663 B,Ring Energy,01PDP,SAN ANDRES,1429.579045,649
1877,42501376010000,"LINESTRING (-102.90883919 33.04218437, -102.90...",MF 732-733 1H,MF 732-733,Amtex Energy,01PDP,SAN ANDRES,NaN,1443
1878,42501376030000,"LINESTRING (-103.006373059 33.082534816, -103....",RED RAIDER 663 C 8H,RED RAIDER 663 C,Ring Energy,01PDP,SAN ANDRES,1452.670508,649


In [ ]:
m, gdf = explore_linestring_clusters(
    df_survey_linestring_with_cluster,
    wkt_col="geom_wkt",
    cluster_col="cluster_id_global",
    tooltip_cols=["uwi", "well_name", "lease_name", "rsv_cat", "avg_hz_spacing_ft", "cluster_id_global"],
    popup_cols=["uwi", "well_name", "lease_name", "operator", "rsv_cat", "bench", "avg_hz_spacing_ft", "cluster_id_global"],
    zoom_start=10,
)

m.save(r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\data\dbscan_cluster_linestrings.html")

## 8. HDBSCAN - Class

### 8.1 Defining DBSCAN class

In [18]:
@dataclass
class LeaseAwareBucketedWellHDBSCAN:
    """
    Lease-aware HDBSCAN bundling with spacing-based buckets + Bayesian optimization.

    High-level:
      - Join midpoints + spacing + header (lease_name).
      - Compute lease-level spacing metric (median of per-well spacing).
      - Bucket leases by that lease-level spacing (explicit bins or bucket_width_ft).
      - For each spacing bucket:
          * Run Bayesian optimization over:
                - min_cluster_size  (via min_cluster_frac * n_bucket)
                - min_samples
          * Run HDBSCAN (haversine metric) to get labels.
      - Optionally:
          * Post-split large clusters into smaller "balls" of radius <= max_bundle_radius_ft.
          * Assign unique cluster IDs to noise wells (-1) so every well has a unique non-negative ID.

    Output columns after fit_predict():
      - cluster_id_hdbscan : raw HDBSCAN labels (noise = -1).
      - cluster_id_global  : after radius-based splitting and optional noise renumbering.
    """

    # Data
    midpoints_df: pd.DataFrame
    spacing_df: pd.DataFrame
    header_df: pd.DataFrame

    # Column names
    uwi_col: str = "well_i"
    spacing_well_col: str = "well_i"
    header_uwi_col: str = "well_i"
    lease_col: str = "lease_name"

    lat_col: str = "latitude"
    lon_col: str = "longitude"
    spacing_col: str = "avg_hz_spacing_ft"
    lease_spacing_col: str = "lease_median_spacing_ft"

    # Bucketing on lease-level spacing
    bucket_width_ft: Optional[float] = 500.0
    spacing_bins_ft: Optional[Sequence[float]] = None
    min_bucket_size: int = 10

    # HDBSCAN hyperparameter search space
    min_cluster_frac_bounds: Tuple[float, float] = (0.02, 0.20)
    min_samples_bounds: Tuple[int, int] = (1, 5)

    random_state: int = 0
    objective_fn: Optional[ScoreFn] = None

    # Post-processing: radius-based splitting + noise handling
    max_bundle_radius_ft: Optional[float] = None
    assign_unique_ids_to_noise: bool = False

    # Filled after fit
    bucket_params_: Dict[int, Dict[str, float]] = field(
        init=False, default_factory=dict
    )

    # ------------------------------------------------------------------ #
    # Internal helpers: data prep & bucketing
    # ------------------------------------------------------------------ #
    def _prepare_dataframe(self) -> pd.DataFrame:
        """
        Join midpoints + spacing + header, and compute lease-level spacing.

        Returns df with:
          - midpoints columns
          - header lease_col
          - per-well spacing_col
          - lease_spacing_col (median spacing per lease)
        """
        # spacing_df + header_df -> per-well spacing + lease
        spacing_with_lease = self.spacing_df.merge(
            self.header_df[[self.header_uwi_col, self.lease_col]],
            left_on=self.spacing_well_col,
            right_on=self.header_uwi_col,
            how="left",
        )

        # lease-level spacing metric (median)
        lease_stats = (
            spacing_with_lease
            .dropna(subset=[self.lease_col, self.spacing_col])
            .groupby(self.lease_col, as_index=False)
            .agg(**{self.lease_spacing_col: (self.spacing_col, "median")})
        )

        # midpoints + header -> get lease for each midpoint well
        df = self.midpoints_df.merge(
            self.header_df[[self.header_uwi_col, self.lease_col]],
            left_on=self.uwi_col,
            right_on=self.header_uwi_col,
            how="left",
        )

        # add per-well spacing
        df = df.merge(
            self.spacing_df[[self.spacing_well_col, self.spacing_col]],
            left_on=self.uwi_col,
            right_on=self.spacing_well_col,
            how="left",
        )

        # add lease-level spacing metric
        df = df.merge(
            lease_stats[[self.lease_col, self.lease_spacing_col]],
            on=self.lease_col,
            how="left",
        )

        return df

    def _assign_buckets(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Assign spacing buckets based on lease-level spacing (lease_spacing_col).

        Priority:
          - If spacing_bins_ft is provided: use pd.cut on lease_spacing_col.
          - Else if bucket_width_ft is None: everyone in bucket 0.
          - Else: bucket_id = floor(lease_spacing_col / bucket_width_ft).
        """
        df = df.copy()

        if self.spacing_bins_ft is not None:
            bucket_codes = pd.cut(
                df[self.lease_spacing_col],
                bins=self.spacing_bins_ft,
                right=False,   # [low, high)
                labels=False,  # 0,1,2,...
            )
            df["spacing_bucket_id"] = bucket_codes.astype("Int64")
            return df

        if self.bucket_width_ft is None:
            df["spacing_bucket_id"] = 0
            return df

        bucket = (
            df[self.lease_spacing_col] / self.bucket_width_ft
        ).floordiv(1).astype("Int64")
        df["spacing_bucket_id"] = bucket
        return df

    # ------------------------------------------------------------------ #
    # HDBSCAN core + Bayesian optimization
    # ------------------------------------------------------------------ #
    @staticmethod
    def _hdbscan_labels(
        lat: np.ndarray,
        lon: np.ndarray,
        min_cluster_size: int,
        min_samples: int,
    ) -> np.ndarray:
        """
        Run HDBSCAN with haversine metric on lat/lon points.
        """
        coords_rad = np.vstack([np.radians(lat), np.radians(lon)]).T

        clusterer = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="haversine",
            core_dist_n_jobs=-1,
        )
        labels = clusterer.fit_predict(coords_rad)
        return labels

    def _optimize_bucket(
        self,
        bucket_df: pd.DataFrame,
    ) -> Tuple[np.ndarray, int, int, float]:
        """
        Bayesian optimization over min_cluster_frac & min_samples for one bucket.

        Returns
        -------
        best_labels : np.ndarray
        best_min_cluster_size : int
        best_min_samples : int
        best_score : float
        """
        n = len(bucket_df)
        lat = bucket_df[self.lat_col].values
        lon = bucket_df[self.lon_col].values

        def objective(min_cluster_frac: float, min_samples: float) -> float:
            min_cluster_size = max(2, int(round(min_cluster_frac * n)))
            min_samples_int = max(1, int(round(min_samples)))

            labels = self._hdbscan_labels(
                lat=lat,
                lon=lon,
                min_cluster_size=min_cluster_size,
                min_samples=min_samples_int,
            )

            if self.objective_fn is not None:
                return float(self.objective_fn(labels, bucket_df))

            # default: maximize number of wells in clusters (non-noise)
            return float((labels != -1).sum())

        pbounds = {
            "min_cluster_frac": self.min_cluster_frac_bounds,
            "min_samples": self.min_samples_bounds,
        }

        optimizer = BayesianOptimization(
            f=objective,
            pbounds=pbounds,
            random_state=self.random_state,
            allow_duplicate_points=True,
        )
        optimizer.maximize()

        best_params = optimizer.max["params"]
        best_score = float(optimizer.max["target"])

        best_min_cluster_size = max(
            2,
            int(round(best_params["min_cluster_frac"] * n)),
        )
        best_min_samples = max(
            1,
            int(round(best_params["min_samples"])),
        )

        best_labels = self._hdbscan_labels(
            lat=lat,
            lon=lon,
            min_cluster_size=best_min_cluster_size,
            min_samples=best_min_samples,
        )

        return best_labels, best_min_cluster_size, best_min_samples, best_score

    # ------------------------------------------------------------------ #
    # Radius-based greedy splitting (Option A) + noise handling
    # ------------------------------------------------------------------ #
    def _greedy_radius_bundles_haversine(
        self,
        coords_deg: np.ndarray,
        max_radius_ft: float,
    ) -> np.ndarray:
        """
        Greedy 'ball' splitting inside one cluster.

        Each bundle is the set of points within max_radius_ft of a seed well
        (in haversine distance). Repeats until all wells are assigned.

        Parameters
        ----------
        coords_deg : array (n, 2)
            [lat, lon] in degrees for ONE cluster.
        max_radius_ft : float
            Max radius from seed to include points in the bundle (feet).

        Returns
        -------
        bundle_ids : np.ndarray (n,)
            Integer bundle ID 0,1,2,... (local to this cluster).
        """
        n = coords_deg.shape[0]
        if n == 0:
            return np.array([], dtype=int)
        if n == 1:
            return np.array([0], dtype=int)

        coords_rad = np.radians(coords_deg)
        R_earth_ft = 6371008.8 * 3.28084  # Earth radius in feet

        remaining = np.arange(n)
        bundle_ids = np.full(n, -1, dtype=int)
        next_bundle_id = 0

        while remaining.size > 0:
            seed_idx = remaining[0]
            seed = coords_rad[seed_idx]
            lat1 = seed[0]
            lon1 = seed[1]

            lat2 = coords_rad[remaining, 0]
            lon2 = coords_rad[remaining, 1]

            dlat = lat2 - lat1
            dlon = lon2 - lon1

            a = (
                np.sin(dlat / 2.0) ** 2
                + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
            )
            a = np.clip(a, 0.0, 1.0)
            c = 2.0 * np.arcsin(np.sqrt(a))
            dist_ft = R_earth_ft * c

            in_ball_mask = dist_ft <= max_radius_ft
            in_ball_idx = remaining[in_ball_mask]

            bundle_ids[in_ball_idx] = next_bundle_id
            next_bundle_id += 1

            remaining = remaining[~in_ball_mask]

        return bundle_ids

    def _split_clusters_by_radius(
        self,
        df: pd.DataFrame,
        cluster_col: str,
    ) -> np.ndarray:
        """
        Post-process clusters: for each HDBSCAN cluster, split into smaller
        bundles such that each bundle fits inside a ball of radius
        max_bundle_radius_ft around some seed.

        Returns
        -------
        new_labels : np.ndarray
            New global cluster labels after radius-based splitting.
        """
        if self.max_bundle_radius_ft is None:
            return df[cluster_col].to_numpy()

        max_r = float(self.max_bundle_radius_ft)
        labels = df[cluster_col].to_numpy().copy()

        existing = labels[labels >= 0]
        if existing.size == 0:
            return labels

        next_label = int(existing.max()) + 1

        for cid in sorted(np.unique(existing)):
            mask = labels == cid
            idx_cluster = np.where(mask)[0]

            if idx_cluster.size <= 1:
                continue

            coords_deg = df.loc[idx_cluster, [self.lat_col, self.lon_col]].to_numpy()
            bundle_ids = self._greedy_radius_bundles_haversine(coords_deg, max_r)
            unique_bundles = np.unique(bundle_ids)

            if unique_bundles.size == 1:
                continue  # whole cluster fits in one ball

            # keep cid for first bundle, assign new ids for others
            for j, b in enumerate(sorted(unique_bundles)):
                idx_bundle = idx_cluster[bundle_ids == b]
                if j == 0:
                    continue  # keep original cid
                labels[idx_bundle] = next_label
                next_label += 1

        return labels

    # ------------------------------------------------------------------ #
    # Public API
    # ------------------------------------------------------------------ #
    def preview_buckets(self) -> pd.DataFrame:
        """
        Summarize spacing_bucket_id distribution BEFORE clustering.

        Useful for tuning bucket_width_ft / spacing_bins_ft / min_bucket_size.
        """
        df = self._prepare_dataframe()
        df = self._assign_buckets(df)

        summary = (
            df.groupby("spacing_bucket_id")
              .agg(
                  n_wells=(self.lease_spacing_col, "size"),
                  lease_spacing_min=(self.lease_spacing_col, "min"),
                  lease_spacing_max=(self.lease_spacing_col, "max"),
                  lease_spacing_mean=(self.lease_spacing_col, "mean"),
              )
              .sort_index()
        )
        return summary

    def fit_predict(self) -> pd.DataFrame:
        """
        Run lease-aware, spacing-bucketed HDBSCAN with Bayesian optimization,
        optional radius splitting, and optional unique IDs for noise.
        """
        df = self._prepare_dataframe()
        df = self._assign_buckets(df)

        global_labels = np.full(len(df), -1, dtype=int)
        global_cluster_offset = 0
        bucket_params: Dict[int, Dict[str, float]] = {}

        for bucket_id, bucket_df in df.groupby("spacing_bucket_id"):
            if pd.isna(bucket_id):
                continue

            n_bucket = len(bucket_df)
            if n_bucket < self.min_bucket_size:
                continue

            (
                labels_local,
                best_min_cluster_size,
                best_min_samples,
                best_score,
            ) = self._optimize_bucket(bucket_df)

            mask_clustered = labels_local != -1
            unique_local = np.unique(labels_local[mask_clustered])
            n_clusters = len(unique_local)
            n_clustered = int(mask_clustered.sum())

            if n_clusters == 0:
                continue

            # map bucket-local labels -> global label space
            label_map = {
                local: (global_cluster_offset + i)
                for i, local in enumerate(sorted(unique_local))
            }

            # start everyone as noise
            remapped = np.full_like(labels_local, -1, dtype=int)

            # assign global ids only for non-noise labels
            for local_label, global_label in label_map.items():
                remapped[labels_local == local_label] = global_label

            global_labels[bucket_df.index.values] = remapped
            global_cluster_offset += n_clusters

            bucket_params[int(bucket_id)] = {
                "min_cluster_size": float(best_min_cluster_size),
                "min_samples": float(best_min_samples),
                "score": float(best_score),
                "n_wells": float(n_bucket),
                "n_clusters": float(n_clusters),
                "n_clustered": float(n_clustered),
            }

        df_out = df.copy()

        # raw HDBSCAN labels (before radius-based splitting)
        df_out["cluster_id_hdbscan"] = global_labels

        # radius-based splitting (if enabled)
        if self.max_bundle_radius_ft is not None:
            labels = self._split_clusters_by_radius(
                df_out,
                cluster_col="cluster_id_hdbscan",
            )
        else:
            labels = global_labels.copy()

        # optionally give unique IDs to noise wells too
        if self.assign_unique_ids_to_noise:
            noise_mask = labels == -1
            if noise_mask.any():
                non_noise = labels[labels >= 0]
                start_id = int(non_noise.max()) + 1 if non_noise.size > 0 else 0
                noise_idx = np.where(noise_mask)[0]
                labels[noise_idx] = np.arange(
                    start_id, start_id + noise_idx.size, dtype=int
                )

        df_out["cluster_id_global"] = labels
        self.bucket_params_ = bucket_params

        return df_out

In [19]:
np.arange(0, 3000, 50)

array([   0,   50,  100,  150,  200,  250,  300,  350,  400,  450,  500,
        550,  600,  650,  700,  750,  800,  850,  900,  950, 1000, 1050,
       1100, 1150, 1200, 1250, 1300, 1350, 1400, 1450, 1500, 1550, 1600,
       1650, 1700, 1750, 1800, 1850, 1900, 1950, 2000, 2050, 2100, 2150,
       2200, 2250, 2300, 2350, 2400, 2450, 2500, 2550, 2600, 2650, 2700,
       2750, 2800, 2850, 2900, 2950])

In [35]:
hdb_lease = LeaseAwareBucketedWellHDBSCAN(
    midpoints_df=df_midpoints_filter,
    spacing_df=df_avg_spacing,
    header_df=df_raw_wellheader,
    uwi_col="uwi",
    spacing_well_col="uwi",
    header_uwi_col="uwi",
    lease_col="lease_name",
    lat_col="mid_Lat",
    lon_col="mid_Lon",
    spacing_col="avg_hz_spacing_ft",
    # bucket_width_ft=500.0,
    spacing_bins_ft=np.arange(0, 3000, 50),
    min_bucket_size=10,
    min_cluster_frac_bounds=(0.02, 0.20),
    min_samples_bounds=(1, 5),
    random_state=42,
    max_bundle_radius_ft=1000.0,       # radius cap for greedy splitting
    assign_unique_ids_to_noise=True,    # give unique IDs to noise
)

In [36]:
hdb_lease.preview_buckets()

,n_wells,lease_spacing_min,lease_spacing_max,lease_spacing_mean
spacing_bucket_id,,,,
0,19,10.079829,49.312984,26.336308
1,2,92.992722,92.992722,92.992722
2,2,145.512404,145.512404,145.512404
4,3,226.341605,236.067348,232.046146
6,5,302.925638,345.774661,322.317960
7,7,364.922868,391.982874,381.871718
8,1,415.918590,415.918590,415.918590
9,12,454.172395,488.016639,467.478951
10,3,535.629615,541.004071,539.212585


In [37]:
df_hdb_clusters_lease_hdbscan = hdb_lease.fit_predict()

|   iter    |  target   | min_cl... | min_sa... |
-------------------------------------------------
| 1         | 17.0      | 0.08742   | 4.803     |
| 2         | 17.0      | 0.1518    | 3.395     |
| 3         | 17.0      | 0.04808   | 1.624     |
| 4         | 17.0      | 0.03046   | 4.465     |
| 5         | 17.0      | 0.1282    | 3.832     |
| 6         | 17.0      | 0.1578    | 1.0       |
| 7         | 17.0      | 0.1997    | 1.002     |
| 8         | 17.0      | 0.03343   | 5.0       |
| 9         | 17.0      | 0.1699    | 1.001     |
| 10        | 17.0      | 0.02294   | 4.999     |
| 11        | 17.0      | 0.1937    | 1.001     |
| 12        | 17.0      | 0.1848    | 4.999     |
| 13        | 19.0      | 0.06775   | 1.0       |
| 14        | 19.0      | 0.03435   | 1.001     |
| 15        | 19.0      | 0.04909   | 1.052     |
| 16        | 19.0      | 0.02      | 1.133     |
| 17        | 19.0      | 0.09078   | 1.178     |
| 18        | 19.0      | 0.02      | 1.244     |


In [38]:
df_hdb_clusters_lease_hdbscan

,uwi,heel_lat,heel_lon,toe_lat,toe_lon,mid_Lat,mid_Lon,lease_name,avg_hz_spacing_ft,lease_median_spacing_ft,spacing_bucket_id,cluster_id_hdbscan,cluster_id_global
0,30025410040100,33.128186,-103.063231,33.139705,-103.063334,33.133946,-103.063283,BROKEN SPOKE 2 STATE,933.507723,1183.017361,23,38,38
1,30025421210000,33.113373,-103.069194,33.125204,-103.069167,33.119289,-103.069181,DOG BAR 11 FEE,1427.412568,1727.793704,34,103,103
2,30025426220000,33.113377,-103.072581,33.125239,-103.073067,33.119308,-103.072824,DOG BAR 11 FEE,1727.793704,1727.793704,34,103,1071
3,30025428730000,33.113636,-103.063432,33.125201,-103.063393,33.119418,-103.063413,DOG BAR 11 FEE,1754.494781,1727.793704,34,103,1072
4,30025435920100,33.097298,-103.074979,33.111496,-103.075177,33.104397,-103.075078,PINKMAN FEE,NaN,NaN,<NA>,-1,1180
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1875,42501375990000,33.080080,-103.003140,33.066330,-103.003103,33.073205,-103.003122,RED RAIDER 663 A,1553.023535,1553.023535,31,93,867
1876,42501376000000,33.080256,-103.005285,33.066333,-103.005288,33.073295,-103.005287,RED RAIDER 663 B,1429.579045,1429.579045,28,87,613
1877,42501376010000,33.043411,-102.905997,33.047070,-102.897379,33.045240,-102.901688,MF 732-733,NaN,NaN,<NA>,-1,1492
1878,42501376030000,33.080130,-103.007432,33.062673,-103.007415,33.071402,-103.007424,RED RAIDER 663 C,1452.670508,1452.670508,29,88,658


In [39]:
df_survey_linestring_with_hdbscan_cluster = df_survey_linestring.merge(df_hdb_clusters_lease_hdbscan[[ "uwi", "avg_hz_spacing_ft", "cluster_id_global"]], on="uwi", how="left").reset_index(drop=True).copy()

In [40]:
m, gdf = explore_linestring_clusters(
    df_survey_linestring_with_hdbscan_cluster,
    wkt_col="geom_wkt",
    cluster_col="cluster_id_global",
    tooltip_cols=["uwi", "well_name", "lease_name", "rsv_cat", "avg_hz_spacing_ft", "cluster_id_global"],
    popup_cols=["uwi", "well_name", "lease_name", "operator", "rsv_cat", "bench", "avg_hz_spacing_ft", "cluster_id_global"],
    zoom_start=10,
)

m.save(r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\data\hdbscan_cluster_linestrings.html")

## 9. Spacing Distance Config

### 9.1 Defining Spacing Distance Config

In [25]:
@dataclass
class SpacingDistanceConfig:
    """
    Configuration knobs for turning WellSpacingCalculator output into a single
    clustering distance per pair.

    Core idea
    =========
    We assume your spacing DataFrame has already been normalized via
    `_apply_effective_horizontal`, so:

      - `horizontal_dist` is the *effective* horizontal spacing:
          * parallel_like  -> crossline mean over the true overlap corridor
          * oblique/perp   -> min_distance_ft (closest approach)
      - `3D_dist` is recomputed from that effective horizontal + `vertical_dist`.

    This config controls how we turn those into `pair_distance`:

      * `use_3d`:
          False -> use horizontal only (map-view spacing).
          True  -> use full 3D distance.

      * `min_parallel_overlap_pct`:
          Minimum average overlap fraction for parallel-like pairs.
          We use `0.5*(overlap_pct_i + overlap_pct_k)` as "coverage".
          Below this, the pair is treated as not informative and excluded
          (`valid_for_clustering = False`).

      * `min_oblique_contact_pct`:
          Minimum `contact_pct_i` (fraction of well_i arc length that is
          within `contact_threshold_ft` of well_k) for oblique/perp pairs.
          Below this, the pair is excluded.

      * `weight_by_coverage`:
          If True:
              pair_distance = base_distance / coverage_used
          If False:
              pair_distance = base_distance

          where:
              - parallel_like -> coverage_used = avg(overlap_pct_i, overlap_pct_k)
              - oblique/perp  -> coverage_used = contact_pct_i

          Intuition:
              A pair that is 600 ft apart but overlapped/contacted for
              80% of its length is "more strongly interacting" than a
              pair that briefly kisses at 600 ft for 5% of its length.
              Dividing by coverage pushes the latter further away in
              clustering metric space.

      * `drop_misaligned`:
          If True:
              `pair_alignment` not in {"parallel_like", "oblique", "perpendicular"}
              are excluded (valid_for_clustering = False).
          If False:
              they are kept if they pass distance filters, but their
              coverage_used is 0 → only meaningful if weight_by_coverage=False.

      * `max_valid_distance_ft`:
          Optional hard cutoff on pair_distance. Any pair with
              pair_distance > max_valid_distance_ft
          is flagged as invalid_for_clustering, even if distance could
          be computed. This is your "beyond any interaction zone" guardrail.

      * `epsilon_cov`:
          Small epsilon to avoid division by zero when weight_by_coverage=True.
    """
    use_3d: bool = False
    min_parallel_overlap_pct: float = 0.10   # e.g. require at least 10% overlap
    min_oblique_contact_pct: float = 0.05    # e.g. require at least 5% contact
    weight_by_coverage: bool = True
    drop_misaligned: bool = True
    max_valid_distance_ft: Optional[float] = None
    epsilon_cov: float = 1e-3


class SpacingDistanceBuilder:
    """
    Turn a WellSpacingCalculator spacing table into a single numeric distance
    per (well_i, well_k) suitable for clustering (DBSCAN / HDBSCAN).

    Expected input
    ==============
    A pandas.DataFrame with one row per pair (well_i, well_k) that has been
    normalized via `_apply_effective_horizontal`, i.e. it contains:

      Required columns
      ----------------
      - 'well_i', 'well_k'
      - 'pair_alignment'         # "parallel_like", "oblique", "perpendicular", ...
      - 'horizontal_dist'        # already effective (crossline or min-distance)
      - 'vertical_dist'
      - '3D_dist'                # effective 3D (after normalization)
      - 'reject_reason'          # empty string "" for accepted pairs

      For parallel-like:
      - 'overlap_pct_i'
      - 'overlap_pct_k'

      For oblique/perpendicular:
      - 'contact_pct_i'          # fraction of i within contact_threshold_ft
      - 'proj_coverage_i_pct'    # optional, not strictly needed here

    Output
    ======
    The `build_pair_distance` method returns a **copy** of the input with
    these extra columns:

      - 'pair_distance' : float
            The clustering distance used downstream. Lower = more tightly
            interacting laterals, higher = weak or distant relationships.

      - 'pair_coverage_used' : float in [0,1]
            The coverage metric actually used:
                parallel_like -> avg(overlap_pct_i, overlap_pct_k)
                oblique/perp  -> contact_pct_i
                others        -> 0.0

      - 'valid_for_clustering' : bool
            True if this pair passed all filters (reject_reason empty,
            coverage above thresholds, distance not NaN, and inside any
            max_valid_distance_ft gate).

      - 'pair_distance_basis' : {"horizontal_dist", "3D_dist"}
            Documents whether we used the 2D or 3D effective metric.

    Usage pattern
    =============
    >>> cfg = SpacingDistanceConfig(
    ...     use_3d=False,
    ...     min_parallel_overlap_pct=0.20,
    ...     min_oblique_contact_pct=0.10,
    ...     weight_by_coverage=True,
    ...     max_valid_distance_ft=2640.0,   # e.g. 1/2 mile cap
    ... )
    >>> builder = SpacingDistanceBuilder(cfg)
    >>> spacing_with_dist = builder.build_pair_distance(spacing_df)

    From there you can:
      - Filter to spacing_with_dist[spacing_with_dist["valid_for_clustering"]]
      - Inspect the histogram of pair_distance
      - Build a graph / adjacency structure for clustering
    """

    def __init__(self, config: Optional[SpacingDistanceConfig] = None) -> None:
        self.config = config or SpacingDistanceConfig()

    def build_pair_distance(self, spacing_df: pd.DataFrame) -> pd.DataFrame:
        """
        Compute `pair_distance` and related flags in a fully vectorized way.

        Parameters
        ----------
        spacing_df : pd.DataFrame
            Output from WellSpacingCalculator (after `_apply_effective_horizontal`).

        Returns
        -------
        pd.DataFrame
            A copy of `spacing_df` with additional columns:
                'pair_distance', 'pair_coverage_used',
                'valid_for_clustering', 'pair_distance_basis'.
        """
        required = {
            "well_i", "well_k",
            "pair_alignment",
            "horizontal_dist", "vertical_dist", "3D_dist",
            "reject_reason",
            "overlap_pct_i", "overlap_pct_k",
            "contact_pct_i",
        }
        missing = required - set(spacing_df.columns)
        if missing:
            raise ValueError(
                f"spacing_df is missing required columns: {sorted(missing)}"
            )

        cfg = self.config
        df = spacing_df.copy()

        # --- Base distance: horizontal-only or 3D (already normalized) ---
        if cfg.use_3d:
            base = df["3D_dist"].to_numpy(dtype="float64")
            basis_label = "3D_dist"
        else:
            base = df["horizontal_dist"].to_numpy(dtype="float64")
            basis_label = "horizontal_dist"

        # --- Alignment masks ---
        align = df["pair_alignment"].astype(str).str.lower()
        is_parallel = align.eq("parallel_like").to_numpy()
        is_oblique = align.eq("oblique").to_numpy()
        is_perp = align.eq("perpendicular").to_numpy()
        is_oblique_or_perp = is_oblique | is_perp

        # --- Coverage metrics ---
        # For parallel-like: average overlap fraction of i and k
        overlap_i = df["overlap_pct_i"].to_numpy(dtype="float64")
        overlap_k = df["overlap_pct_k"].to_numpy(dtype="float64")
        cov_parallel = 0.5 * (np.nan_to_num(overlap_i, nan=0.0) +
                              np.nan_to_num(overlap_k, nan=0.0))

        # For oblique/perp: use contact fraction along i
        contact_i = df["contact_pct_i"].to_numpy(dtype="float64")
        cov_oblique = np.nan_to_num(contact_i, nan=0.0)

        n = len(df)
        coverage_used = np.zeros(n, dtype="float64")

        # Assign coverage_used by alignment
        coverage_used[is_parallel] = cov_parallel[is_parallel]
        coverage_used[is_oblique_or_perp] = cov_oblique[is_oblique_or_perp]

        # --- Base validity: numeric distances + no explicit reject_reason ---
        reject = df["reject_reason"].fillna("").astype(str).to_numpy()
        has_explicit_reject = reject != ""
        base_valid = np.isfinite(base) & (~has_explicit_reject)

        # --- Alignment-specific coverage thresholds ---
        valid_parallel = is_parallel & (cov_parallel >= float(cfg.min_parallel_overlap_pct))
        valid_oblique = is_oblique_or_perp & (cov_oblique >= float(cfg.min_oblique_contact_pct))

        # Misaligned: everything else
        is_misaligned = ~(is_parallel | is_oblique_or_perp)

        if cfg.drop_misaligned:
            valid_misaligned = np.zeros(n, dtype=bool)
        else:
            # Allow them through only if base distance is finite.
            valid_misaligned = is_misaligned & np.isfinite(base)

        valid_alignment = valid_parallel | valid_oblique | valid_misaligned

        # --- Combine all validity conditions ---
        valid = base_valid & valid_alignment

        # --- Compute pair_distance, optionally coverage-weighted ---
        pair_distance = np.full(n, np.nan, dtype="float64")
        if cfg.weight_by_coverage:
            denom = np.maximum(coverage_used, float(cfg.epsilon_cov))
            pair_distance[valid] = base[valid] / denom[valid]
        else:
            pair_distance[valid] = base[valid]

        # --- Optional hard cutoff on distance ---
        if cfg.max_valid_distance_ft is not None:
            too_far = pair_distance > float(cfg.max_valid_distance_ft)
            # Only affect rows that already had a distance
            valid[too_far & np.isfinite(pair_distance)] = False

        # Optionally you could add a min distance cutoff here as well if desired.

        # --- Write back to DataFrame ---
        df["pair_distance"] = pair_distance
        df["pair_coverage_used"] = coverage_used
        df["valid_for_clustering"] = valid
        df["pair_distance_basis"] = basis_label

        return df

In [23]:
folder_path = r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\notebooks\SpacingStats"
parquet_files = glob.glob(f'{folder_path}/*.parquet') 

df_spacing = pd.concat((pd.read_parquet(file) for file in parquet_files), ignore_index=True)

df_spacing_filtered = df_spacing[df_spacing["reject_reason"]==""].reset_index(drop=True).copy()

In [36]:
cfg = SpacingDistanceConfig(
    use_3d=False,                  # map-view spacing
    min_parallel_overlap_pct=0.20, # require at least 20% mutual overlap
    min_oblique_contact_pct=0.10,  # require 10% contact along i
    weight_by_coverage=True,
    max_valid_distance_ft=2640.0,  # ignore > 0.5 mile relationships
)
builder = SpacingDistanceBuilder(cfg)
spacing_with_dist = builder.build_pair_distance(df_spacing_filtered)

# Filter to only usable edges:
edges = spacing_with_dist[spacing_with_dist["valid_for_clustering"] & spacing_with_dist["well_i"].isin(df_raw_wellheader[df_raw_wellheader["lease_name"]=="WADDELL TR A"]["uwi"].values)].copy()

In [37]:
edges

,well_i,well_k,horizontal_dist,horizontal_dist_median,vertical_dist,3D_dist,drill_direction_i,drill_direction_k,n_samples,dy_p5,...,contact_len_i_interior_ft_T300,contact_pct_i_interior_T300,horizontal_crossline_mean_ft,hz_effective,hz_basis,3D_dist_effective,pair_distance,pair_coverage_used,valid_for_clustering,pair_distance_basis
8773,42103365150000,42103368020000,1553.588513,1566.306155,19.0555,1553.705371,NS,NS,46.0,1346.010849,...,NaN,NaN,1553.588513,1553.588513,crossline_mean,1553.705371,1810.465698,0.858115,True,horizontal_dist
8775,42103365150000,42103368920000,2271.417898,2271.752862,1094.7520,2521.472032,NS,NS,46.0,2163.216558,...,NaN,NaN,2271.417898,2271.417898,crossline_mean,2521.472032,2479.589277,0.916046,True,horizontal_dist
8777,42103365150000,42103369150000,1295.834461,1294.305539,198.1105,1310.890811,NS,NS,46.0,1160.243434,...,NaN,NaN,1295.834461,1295.834461,crossline_mean,1310.890811,1474.948686,0.878562,True,horizontal_dist
8778,42103365150000,42103369230000,395.490518,405.988865,354.4475,531.079825,NS,NS,46.0,320.017034,...,NaN,NaN,395.490518,395.490518,crossline_mean,531.079825,457.894643,0.863715,True,horizontal_dist
8782,42103365150000,42103370430000,894.535046,902.073264,1126.3720,1438.369504,NS,NS,46.0,753.755502,...,NaN,NaN,894.535046,894.535046,crossline_mean,1438.369504,1011.026900,0.884779,True,horizontal_dist
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16030,42103375480000,42103373880000,2246.582516,2237.754667,336.0950,2271.583776,NS,NS,82.0,2024.641694,...,NaN,NaN,2246.582516,2246.582516,crossline_mean,2271.583776,2266.356366,0.991275,True,horizontal_dist
16039,42103375480000,42103375430000,1882.296963,1876.111140,38.3550,1882.687697,NS,NS,82.0,1609.716141,...,NaN,NaN,1882.296963,1882.296963,crossline_mean,1882.687697,1903.313597,0.988958,True,horizontal_dist
16042,42103375480000,42103375470000,734.868143,741.288163,28.2550,735.411131,NS,NS,82.0,719.858569,...,NaN,NaN,734.868143,734.868143,crossline_mean,735.411131,735.791375,0.998745,True,horizontal_dist
16045,42103375490000,42103369030000,2318.854425,2325.978067,289.0965,2336.806075,NS,NS,64.0,2178.918744,...,NaN,NaN,2318.854425,2318.854425,crossline_mean,2336.806075,2450.934308,0.946110,True,horizontal_dist


In [31]:
@dataclass
class PairMatrixConfig:
    """
    Configuration for turning a pairwise spacing table into a dense distance
    matrix suitable for metric='precomputed' clustering.

    Parameters
    ----------
    id_col_i : str
        Column name for the "source" well ID. Default 'well_i'.
    id_col_k : str
        Column name for the "target" well ID. Default 'well_k'.
    distance_col : str
        Column name for the scalar distance you want to cluster on.
        Typically 'pair_distance' produced by SpacingDistanceBuilder.
    valid_col : str
        Boolean column flagging which rows should be used to populate
        the matrix (True = usable edge).
    fill_diagonal : float
        Value to use on the diagonal. For distances, this should be 0.0.
    fill_offdiag_default : Optional[float]
        Value for "no edge" pairs. If None, use a large number:
            max_distance * 10.0

        DBSCAN/HDBSCAN only care about whether distance <= eps or not,
        so "very large" is effectively "no neighbor".
    """

    id_col_i: str = "well_i"
    id_col_k: str = "well_k"
    distance_col: str = "pair_distance"
    valid_col: str = "valid_for_clustering"
    fill_diagonal: float = 0.0
    fill_offdiag_default: Optional[float] = None


class PairDistanceMatrixBuilder:
    """
    Build a dense symmetric distance matrix from a pairwise edges table.

    Typical pipeline
    ----------------
    1) Start with WellSpacingCalculator output (one row per (well_i, well_k)).
    2) Run SpacingDistanceBuilder to add:
         - 'pair_distance'
         - 'valid_for_clustering'
    3) Use this builder to create:

         - dist_matrix : np.ndarray of shape (N, N)
         - ids         : list of well IDs in the same order as dist_matrix
         - id_to_idx   : dict mapping well ID -> row/col index

       Then pass dist_matrix into DBSCAN / HDBSCAN with metric='precomputed'.

    Notes
    -----
    - The matrix is **dense**, so N should be moderate per run
      (e.g., per operator, per bench, per lease cluster).
    - If multiple rows exist for the same (well_i, well_k), we keep the
      **minimum** distance (strongest interaction).
    """

    def __init__(self, config: Optional[PairMatrixConfig] = None) -> None:
        self.config = config or PairMatrixConfig()

    def build_square_matrix(
        self,
        edges: pd.DataFrame,
        *,
        restrict_to_valid: bool = True,
    ) -> Tuple[np.ndarray, List[Any], Dict[Any, int]]:
        """
        Build a symmetric dense distance matrix from the edges table.

        Parameters
        ----------
        edges : pd.DataFrame
            Pairwise spacing table with at least:
              - id_col_i (e.g. 'well_i')
              - id_col_k (e.g. 'well_k')
              - distance_col (e.g. 'pair_distance')
              - valid_col (e.g. 'valid_for_clustering')

        restrict_to_valid : bool, default True
            If True, only edges with valid_col == True and finite distance
            are used. Others are ignored and left as the default off-diagonal
            fill_value.

        Returns
        -------
        dist_matrix : np.ndarray of shape (N, N)
            Symmetric dense distance matrix; diagonal = fill_diagonal.
        ids : list
            List of well IDs, same order as matrix rows/cols.
        id_to_idx : dict
            Mapping from well ID to matrix index.
        """
        cfg = self.config

        required = {cfg.id_col_i, cfg.id_col_k, cfg.distance_col}
        if restrict_to_valid:
            required.add(cfg.valid_col)
        missing = required - set(edges.columns)
        if missing:
            raise ValueError(f"edges is missing required columns: {sorted(missing)}")

        df = edges.copy()

        # Filter rows if requested
        if restrict_to_valid:
            mask_valid = df[cfg.valid_col].astype(bool).to_numpy()
        else:
            # use all rows; treat invalid distances as NaN so they won't write into matrix
            mask_valid = np.ones(len(df), dtype=bool)

        # Distance array
        dist = df[cfg.distance_col].to_numpy(dtype="float64")
        mask_finite = np.isfinite(dist)
        mask = mask_valid & mask_finite

        df = df.loc[mask].copy()
        dist = dist[mask]

        # Build the universe of wells participating in these edges
        ids_i = df[cfg.id_col_i].to_numpy()
        ids_k = df[cfg.id_col_k].to_numpy()
        all_ids = pd.Index(ids_i).append(pd.Index(ids_k)).unique().tolist()

        n = len(all_ids)
        if n == 0:
            raise ValueError("No wells found after filtering; check your masks/criteria.")

        id_to_idx = {wid: idx for idx, wid in enumerate(all_ids)}

        # Decide default off-diagonal fill
        if cfg.fill_offdiag_default is not None:
            default_offdiag = float(cfg.fill_offdiag_default)
        else:
            # big number = "effectively no neighbor"
            max_dist = float(np.nanmax(dist)) if dist.size else 1.0
            default_offdiag = max_dist * 10.0

        # Initialize dense matrix
        mat = np.full((n, n), default_offdiag, dtype="float64")
        np.fill_diagonal(mat, float(cfg.fill_diagonal))

        # Populate from edges; ensure symmetry (i,k) == (k,i),
        # and keep the minimum distance if duplicates exist.
        for (wi, wk, d) in zip(ids_i, ids_k, dist):
            i = id_to_idx[wi]
            k = id_to_idx[wk]
            if i == k:
                # Shouldn't happen, but just in case
                mat[i, i] = min(mat[i, i], d)
            else:
                # keep the strongest (smallest) connection
                if d < mat[i, k]:
                    mat[i, k] = d
                    mat[k, i] = d

        return mat, all_ids, id_to_idx

    @staticmethod
    def to_condensed(square: np.ndarray) -> np.ndarray:
        """
        Convert a square symmetric distance matrix into a condensed
        vector form (upper triangle, i<j) like scipy.spatial.distance.squareform.

        Useful if you want to feed the distances into algorithms that
        expect a 1D condensed matrix.

        Parameters
        ----------
        square : np.ndarray, shape (N, N)
            Symmetric distance matrix.

        Returns
        -------
        condensed : np.ndarray, shape (N*(N-1)//2,)
        """
        if square.ndim != 2 or square.shape[0] != square.shape[1]:
            raise ValueError("square must be a 2D square matrix")
        n = square.shape[0]
        i, j = np.triu_indices(n, k=1)
        return square[i, j]

In [38]:
pm_cfg = PairMatrixConfig(
    id_col_i="well_i",
    id_col_k="well_k",
    distance_col="pair_distance",
    valid_col="valid_for_clustering",
)

pm_builder = PairDistanceMatrixBuilder(pm_cfg)

dist_matrix, well_ids, id_to_idx = pm_builder.build_square_matrix(edges)
print(dist_matrix.shape)  # (N, N)
print(len(well_ids), "unique wells")

(177, 177)
177 unique wells


In [44]:
# Choose an eps in *same units* as pair_distance (feet)
eps_ft = 600.0  # e.g., 1/4 mile interaction radius
min_samples = 2  # min wells per cluster core

db = DBSCAN(
    eps=eps_ft,
    min_samples=min_samples,
    metric="precomputed",
)

labels = db.fit_predict(dist_matrix)

# Build a small cluster assignment table
clusters_df = pd.DataFrame({
    "uwi": well_ids,
    "cluster_dbscan": labels,
})

In [45]:
clusters_df

,uwi,cluster_dbscan
0,42103365150000,0
1,42103365220000,0
2,42103368010000,1
3,42103368020000,0
4,42103368030000,2
...,...,...
172,42103376110000,-1
173,42103376120000,-1
174,42103376130000,-1
175,42103376140000,-1


In [46]:
df_survey_linestring_with_dbscan_cluster_v2 = df_survey_linestring.merge(clusters_df, on="uwi", how="left").reset_index(drop=True).copy()

In [47]:
m, gdf = explore_linestring_clusters(
    df_survey_linestring_with_dbscan_cluster_v2,
    wkt_col="geom_wkt",
    cluster_col="cluster_dbscan",
    tooltip_cols=["uwi", "well_name", "lease_name", "rsv_cat", "cluster_dbscan"],
    popup_cols=["uwi", "well_name", "lease_name", "operator", "rsv_cat", "bench", "cluster_dbscan"],
    zoom_start=10,
)

m.save(r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\data\dbscan_cluster_linestrings_v2.html")